# LlamaIndex ReAct Agent with Long-Term Memory

Build a ReAct agent with LlamaIndex that remembers user preferences and past interactions across conversations using Hindsight memory.

This notebook demonstrates two patterns:
- **Tool Spec pattern**: Use `HindsightToolSpec` directly for full control over tool creation
- **Factory pattern**: Use `create_hindsight_tools()` for a simpler API with include/exclude flags

## Prerequisites

- Python 3.10+
- A [Hindsight Cloud](https://ui.hindsight.vectorize.io/signup) account **or** a self-hosted Hindsight instance
- An OpenAI API key (or any LlamaIndex-supported model)

### Option A: Hindsight Cloud

Sign up at [ui.hindsight.vectorize.io](https://ui.hindsight.vectorize.io/signup) and grab your API key from the dashboard.

### Option B: Self-Hosted (Docker)

```bash
export OPENAI_API_KEY=your-key

docker run --rm -it --pull always -p 8888:8888 -p 9999:9999 \
  -e HINDSIGHT_API_LLM_API_KEY=$OPENAI_API_KEY \
  -e HINDSIGHT_API_LLM_MODEL=gpt-4o-mini \
  -v $HOME/.hindsight-docker:/home/hindsight/.pg0 \
  ghcr.io/vectorize-io/hindsight:latest
```

## Installation

In [ ]:
!pip install hindsight-llamaindex llama-index-llms-openai llama-index-core python-dotenv -U

## Setup

Configure the Hindsight client. Set `HINDSIGHT_API_URL` and `HINDSIGHT_API_KEY` in your `.env` file or environment:

| | Hindsight Cloud | Self-Hosted |
|---|---|---|
| `HINDSIGHT_API_URL` | `https://api.hindsight.vectorize.io` | `http://localhost:8888` |
| `HINDSIGHT_API_KEY` | Your cloud API key | *(not required)* |

> **Note:** You also need `OPENAI_API_KEY` set in your environment or `.env` file for the LlamaIndex model.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

HINDSIGHT_API_URL = os.getenv("HINDSIGHT_API_URL", "http://localhost:8888")
HINDSIGHT_API_KEY = os.getenv("HINDSIGHT_API_KEY")  # Required for Hindsight Cloud

from hindsight_client import Hindsight

client_kwargs = {"base_url": HINDSIGHT_API_URL}
if HINDSIGHT_API_KEY:
    client_kwargs["api_key"] = HINDSIGHT_API_KEY

client = Hindsight(**client_kwargs)

## Pattern 1: Tool Spec — Full Control

Use `HindsightToolSpec` directly with LlamaIndex's native `BaseToolSpec` pattern.
The agent gets retain/recall/reflect tools and decides when to use memory.

### Create the Agent

In [ ]:
from hindsight_llamaindex import HindsightToolSpec
from llama_index.llms.openai import OpenAI
from llama_index.core.agent import ReActAgent


def create_agent(user_id: str) -> ReActAgent:
    """Create a ReAct agent with per-user memory."""
    spec = HindsightToolSpec(
        client=client,
        bank_id=f"user-{user_id}",
        tags=["source:chat"],
        budget="mid",
    )
    tools = spec.to_tool_list()

    return ReActAgent.from_tools(
        tools,
        llm=OpenAI(model="gpt-4o-mini"),
        system_prompt=(
            "You are a helpful assistant with long-term memory. "
            "Use retain_memory to store important facts about the user. "
            "Use recall_memory to search your memory before answering. "
            "Use reflect_on_memory for thoughtful summaries of what you know."
        ),
        verbose=True,
    )

### Create the Memory Bank

In [ ]:
client.create_bank("user-alice", name="Alice's Memory")
print("Bank created: user-alice")

### Conversation 1: Store Preferences

The agent should recognize important facts and store them using `retain_memory`.

In [ ]:
agent = create_agent("alice")
response = agent.chat(
    "Hi! I'm Alice. I'm a data scientist who works with Python and SQL. "
    "I prefer dark mode and use VS Code. Please remember all of this."
)
print(f"\nAgent: {response}")

In [ ]:
import time

# Hindsight processes retained content asynchronously (extracting facts, entities, embeddings).
# The sleep gives the server time to finish before we recall.
time.sleep(3)

### Conversation 2: Recall Preferences

A new agent instance — but memory persists. The agent should search memory to answer.

In [ ]:
agent = create_agent("alice")
response = agent.chat("What IDE do I use? And what's my job?")
print(f"\nAgent: {response}")

### Conversation 3: Reflect on Knowledge

The agent uses `reflect_on_memory` to synthesize a thoughtful answer from accumulated memories.

In [ ]:
agent = create_agent("alice")
response = agent.chat(
    "Based on everything you know about me, what tools and setup "
    "would you recommend for a new machine learning project?"
)
print(f"\nAgent: {response}")

## Pattern 2: Factory — Selective Tools

Use `create_hindsight_tools()` for a simpler API. You can include/exclude specific tools
and pass all configuration in one call.

In [ ]:
from hindsight_llamaindex import create_hindsight_tools

# Create only retain + recall tools (no reflect)
tools = create_hindsight_tools(
    client=client,
    bank_id="user-alice",
    tags=["source:chat"],
    budget="mid",
    include_reflect=False,
)

print(f"Tools created: {[t.metadata.name for t in tools]}")

agent = ReActAgent.from_tools(
    tools,
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="You are a helpful assistant with long-term memory.",
    verbose=True,
)

In [ ]:
response = agent.chat("I also enjoy hiking on weekends. Please remember that.")
print(f"\nAgent: {response}")

In [ ]:
time.sleep(3)

response = agent.chat("What do you know about my hobbies?")
print(f"\nAgent: {response}")

## Selective Tools via `to_tool_list()`

You can also use the tool spec's `to_tool_list(spec_functions=...)` for fine-grained control:

In [ ]:
spec = HindsightToolSpec(client=client, bank_id="user-alice")

# Only expose recall and reflect — read-only memory access
read_only_tools = spec.to_tool_list(
    spec_functions=["recall_memory", "reflect_on_memory"]
)

print(f"Read-only tools: {[t.metadata.name for t in read_only_tools]}")

## Cleanup

In [ ]:
client.delete_bank("user-alice")
print("Bank deleted.")

## Key Takeaways

- **Tool Spec pattern**: Use `HindsightToolSpec` with `to_tool_list()` for native LlamaIndex integration with full `BaseToolSpec` features.
- **Factory pattern**: Use `create_hindsight_tools()` for quick setup with include/exclude flags.
- **Selective tools**: Filter tools via `spec_functions=[...]` or `include_retain/recall/reflect` flags.
- **Per-user banks**: Use `bank_id=f"user-{user_id}"` for per-user memory isolation.
- **Tags**: Scope memories by source, conversation, or topic for precise recall.